In [ ]:
# to use it on sciencecluster, just type source activate numrefields, ipython (ITerm) and then copy commands into CL
import glob
import re
import pandas as pd
from os import listdir
import os.path as op

bids_folder = '/scratch/mrenke/ds-stressrisk'
bids_folder = '/shares/zne.uzh/mrenke/ds-stressrisk'

subFolders = [f for f in listdir(bids_folder) if f[0:3] == 'sub']
subs = []
sessions = []
for e in subFolders:
    if len(e) == 6:
        subs.append(int(e[4:]))
        sessions.append(int(1))
        subs.append(int(e[4:]))
        sessions.append(int(2))

subs.sort()
df_ = pd.DataFrame(data = {'subject':subs, 'session':sessions})

df_raw = df_.set_index(['subject', 'session'])

In [ ]:

def get_files_df(fns, reg):
        data = []

        for fn in fns:
            try:
                data.append(reg.match(fn).groupdict())
                data[-1]['fn'] = fn
            except Exception as e:
                print(f'Problem with {fn}: {e}')

        data = pd.DataFrame(data)
        data['subject'] = [ int(x) for x in data['subject'] ]
        data['session'] = [ int(x) for x in data['session'] ]
        data = data.set_index(['subject', 'session'])
        return data
    

In [ ]:
key = 'glm_stim1.denoise.retroicor'
ses = '/ses-*'
file_format = '.nii.gz'
fns = glob.glob(f'{bids_folder}/derivatives/{key}/sub-*{ses}/func/*{file_format}') 
reg = re.compile('.*/sub-(?P<subject>[0-9]+)_ses-(?P<session>[0-9]+)_task-risk_space-T1w_desc-stims1_pe.nii.gz') # glm denoise model
data_glm = get_files_df(fns, reg)
data_glm = data_glm.rename(columns={'fn':'glm_fn'})

key = 'encoding_model.cv.denoise'
ses = '/ses-*'
file_format = '.nii.gz'
fns = glob.glob(f'{bids_folder}/derivatives/{key}/sub-*{ses}/func/*{file_format}') 
reg = re.compile('.*/sub-(?P<subject>[0-9]+)_ses-(?P<session>[0-9]+)_run-(?P<run>[0-9]+)_desc-r2.optim_space-T1w_pars.nii.gz') # encoding model
reg = re.compile('.*/sub-(?P<subject>[0-9]+)_ses-(?P<session>[0-9]+)_run-1_desc-r2.optim_space-T1w_pars.nii.gz')
reg = re.compile('.*/sub-(?P<subject>[0-9]+)_ses-(?P<session>[0-9]+)_desc-r2.optim_space-T1w_pars.nii.gz') # no CV 
data_encode_cv = get_files_df(fns, reg)
data_encode_cv = data_encode_cv.rename(columns={'fn':'encod_fn'})

key = 'decoded_pdfs.volume.denoise'
ses = ''
file_format = '.tsv'
fns = glob.glob(f'{bids_folder}/derivatives/{key}/sub-*{ses}/func/*{file_format}') 
reg = re.compile('.*/sub-(?P<subject>[0-9]+)_ses-(?P<session>[0-9]+)_mask-NPC_R_nvoxels-250_space-T1w_pars.tsv') # decoding
data_decode = get_files_df(fns, reg)
data_decode = data_decode.rename(columns={'fn':'decode_fn'})


In [ ]:
df = df_raw.join(data_glm)
df = df.join(data_encode_cv)
df = df.join(data_decode)

df.to_csv('/home/mrenke/git/stress_risk/stress_risk/prepare/genModel_steps_files.csv')


In [ ]:
# fmriprep
key = 'fmriprep'
ses = '/ses-*'
file_format = '.nii.gz'
fns = glob.glob(op.join(bids_folder,f'derivatives/{key}/sub-*{ses}/func/*{file_format}'))
reg = re.compile('.*/sub-(?P<subject>[0-9]+)_ses-(?P<session>[0-9]+)_task-risk_run-(?P<run>[0-9]+)_space-T1w_desc-preproc_bold.nii.gz') # fmriprep output
#reg = re.compile('.*/sub-(?P<subject>[0-9]+)_ses-(?P<session>[0-9]+)_task-risk_run-(?P<run>[0-9]+)_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz') # fmriprep output

data_preproc = get_files_df(fns, reg)
data_preproc = data_preproc.rename(columns={'fn':'preproc_fn'})

data_preproc = data_preproc.set_index('run', append= True).unstack('run')
df = df_raw.join(data_preproc)
df.to_csv('/home/mrenke/git/stress_risk/stress_risk/prepare/preproc_files.csv')


# behavior
file_format = '.tsv'
fns = glob.glob(f'/scratch/mrenke/ds-stressrisk/sub-*/ses-*/func/*{file_format}') 
reg = re.compile('.*/sub-(?P<subject>[0-9]+)_ses-(?P<session>[0-9]+)_task-risk_run-(?P<run>[0-9]+)_(events.tsv)') # behav files
data_behav = get_files_df(fns, reg)
data_behav = data_behav.rename(columns={'fn':'behav_fn'})

df = df_raw.join(data_behav)
df.to_csv('/home/mrenke/git/stress_risk/stress_risk/prepare/behav_sciencecluster2_files.csv')



In [ ]:
# get missing subs for sbatch --array=  ..... submit-x-.py

dd = data_glm.unstack('session') # sub 54, ses 2 missing
#dd = data_encode_cv.unstack('session') # sub 54, ses 2 missing

ddf = df_raw.unstack('session')
dd.index

l = str()
for i in ddf.index :
    print(i)
    if i in dd.index:
        print(i)
    else:
        sub = '%02d' % int(i)
        l = l + sub +  ','

l # = 03,05,09,10,14,16,17,18,21,26,31,32,34,35,37,41,42,46,48,49,51,55,57,58,54
    